# Solution — AI 기반 지반함몰 위험도 예측 실습

Beginner, Intermediate, Advanced 과정의 완성 코드와 결과 해석 예시입니다.

먼저 선택한 난이도의 노트북을 수행한 뒤 복습과 검증에 사용하세요.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
candidates = [
    Path("../data/hannam_sinkhole_data.xlsx"),
    Path("data/hannam_sinkhole_data.xlsx"),
    Path("hannam_sinkhole_data.xlsx"),
]
DATA_PATH = next((p for p in candidates if p.exists()), None)
DATA_URL = (
    "https://raw.githubusercontent.com/GeoSeonghoHong/"
    "Seongho-Hong/main/data/hannam_sinkhole_data.xlsx"
)

if DATA_PATH is not None:
    print("로컬 데이터 사용:", DATA_PATH)
    df = pd.read_excel(DATA_PATH, sheet_name="training_data")
else:
    print("로컬 파일이 없어 GitHub에서 데이터를 불러옵니다.")
    df = pd.read_excel(DATA_URL, sheet_name="training_data")

print("shape:", df.shape)
df.head()

## Step 1. 데이터 구조 확인

결측치와 중복, 위험도 분포를 확인합니다. 데이터 품질을 직접 복구하는 활동은 이번 60분 실습 범위에서 제외합니다.

In [ ]:
print(df.info())
print("결측치:", int(df.isna().sum().sum()))
print("중복 행:", int(df.duplicated().sum()))
display(df["risk"].value_counts().sort_index().rename(index={0:"Low", 1:"Medium", 2:"High"}))

In [ ]:
risk_counts = df["risk"].value_counts().sort_index()
ax = risk_counts.plot(kind="bar", color=["#4CAF50", "#FFC107", "#F44336"])
ax.set_title("Risk class distribution")
ax.set_xlabel("risk")
ax.set_ylabel("count")
plt.show()

In [ ]:
def evaluate_model(name, features):
    X = df[features].copy()
    y = df["risk"].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    model = XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "model_object": model,
        "features": features,
        "y_test": y_test,
        "pred": pred,
    }
    return result

## Step 2~4. 세 가지 입력 변수 구성

In [ ]:
water_features = [f"water_Y_{year}" for year in range(5, 60, 5)]
sewer_features = [f"sewer_Y_{year}" for year in range(5, 25, 5)]

basic_features = ["water_Density", "sewer_total_density_km_km2"]
raw_features = water_features + ["water_Density"] + sewer_features + [
    "sewer_total_density_km_km2", "cavity_count"
]
derived_features = [
    "total_water_length", "total_sewer_length", "pipe_total_density",
    "water_sewer_ratio", "old_pipe_ratio", "recent_pipe_ratio",
    "cavity_density_interaction",
]
engineered_features = raw_features + derived_features

In [ ]:
result_a = evaluate_model("Model A: density only", basic_features)
result_b = evaluate_model("Model B: raw features", raw_features)
result_c = evaluate_model("Model C: engineered features", engineered_features)

comparison = pd.DataFrame([
    {"model": r["model"], "accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
    for r in [result_a, result_b, result_c]
]).set_index("model")
display(comparison.round(3))

## 성능 비교

Accuracy는 전체 예측 중 정답의 비율이고, Macro F1-score는 세 위험등급의 F1-score를 동일한 비중으로 평균한 값입니다. 위험등급별 데이터 수가 다르므로 두 지표를 함께 확인합니다.

In [ ]:
comparison.plot(kind="bar", ylim=(0, 1), figsize=(9, 4), color=["#2F75B5", "#70AD47"])
plt.xticks(rotation=15, ha="right")
plt.ylabel("score")
plt.tight_layout()
plt.show()

In [ ]:
best_result = max([result_a, result_b, result_c], key=lambda r: r["macro_f1"])
print("Best model:", best_result["model"])
cm = confusion_matrix(best_result["y_test"], best_result["pred"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Low","Medium","High"], yticklabels=["Low","Medium","High"])
plt.title(best_result["model"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
print(classification_report(best_result["y_test"], best_result["pred"], target_names=["Low","Medium","High"]))

## 변수 중요도

In [ ]:
importance = pd.DataFrame({
    "feature": result_c["features"],
    "importance": result_c["model_object"].feature_importances_,
}).sort_values("importance", ascending=False)
display(importance.head(15))
importance.head(15).sort_values("importance").plot(kind="barh", x="feature", y="importance", figsize=(8, 6), legend=False)
plt.tight_layout()
plt.show()

## 해석 예시

- Model A는 밀집도만 사용하므로 간단한 기준선 역할을 합니다.
- Model B에서 성능이 달라진다면 시공 시기별 관로 길이와 공동 개수가 추가 정보를 제공한 것입니다.
- Model C의 성능이 향상된다면 관로 총길이, 노후도, 공동과 밀집도의 상호작용처럼 도메인 지식을 반영한 표현이 도움이 된 것으로 해석할 수 있습니다.
- High 등급의 Recall이 낮다면 실제 고위험 지역을 놓칠 수 있으므로 전체 Accuracy가 높더라도 주의해야 합니다.
- 변수 중요도는 모델 예측에 사용된 정도를 나타낼 뿐, 해당 변수가 지반함몰의 직접적인 원인임을 증명하지는 않습니다.